# Customer Engagement & Product Utilization Analytics for Retention Strategy

**Dataset:** European_Bank.csv  
**Business goal:** Understand customer engagement, product utilization, and churn/exit patterns, then convert the findings into actionable retention strategies.

### Project questions
1. What is the overall customer exit/churn rate?
2. Which countries have the highest retention risk?
3. Does customer activity affect churn?
4. How does the number of products relate to churn?
5. Which age groups are most at risk?
6. Which customer segments should receive retention campaigns first?
7. What business actions can improve retention?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

df = pd.read_csv('European_Bank.csv')
df.head()


## 1. Data Understanding & Quality Check

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Unique customers:", df['CustomerId'].nunique())


## 2. KPI Summary

In [ ]:
total_customers = len(df)
exited_customers = df['Exited'].sum()
retained_customers = (df['Exited'] == 0).sum()
exit_rate = df['Exited'].mean()

kpis = pd.DataFrame({
    'KPI': ['Total Customers', 'Exited Customers', 'Retained Customers', 'Overall Exit Rate',
            'Active Members', 'Customers with Credit Card'],
    'Value': [
        total_customers,
        exited_customers,
        retained_customers,
        f"{exit_rate:.2%}",
        df['IsActiveMember'].sum(),
        df['HasCrCard'].sum()
    ]
})
kpis


## 3. Exploratory Analysis

In [ ]:
# Country-wise churn
geo = df.groupby('Geography')['Exited'].agg(['count', 'sum', 'mean']).sort_values('mean', ascending=False)
geo['mean'] = geo['mean'].map(lambda x: f'{x:.2%}')
geo


In [ ]:
# Activity-wise churn
activity = df.groupby('IsActiveMember')['Exited'].agg(['count', 'sum', 'mean'])
activity['Activity'] = activity.index.map({0: 'Inactive', 1: 'Active'})
activity['mean'] = activity['mean'].map(lambda x: f'{x:.2%}')
activity[['Activity', 'count', 'sum', 'mean']]


In [ ]:
# Product utilization and churn
products = df.groupby('NumOfProducts')['Exited'].agg(['count', 'sum', 'mean'])
products['mean'] = products['mean'].map(lambda x: f'{x:.2%}')
products


In [ ]:
# Age segmentation
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[17, 25, 35, 45, 55, 65, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66+']
)

age_analysis = df.groupby('AgeGroup', observed=True)['Exited'].agg(['count', 'sum', 'mean'])
age_analysis['mean'] = age_analysis['mean'].map(lambda x: f'{x:.2%}')
age_analysis


## 4. Engagement & Product Utilization Cross-Analysis

In [ ]:
# Country + activity
country_activity = df.groupby(['Geography', 'IsActiveMember'])['Exited'].agg(['count', 'mean'])
country_activity['Activity'] = country_activity.index.get_level_values(1).map({0:'Inactive', 1:'Active'})
country_activity['mean'] = country_activity['mean'].map(lambda x: f'{x:.2%}')
country_activity[['Activity', 'count', 'mean']]


In [ ]:
# Product count + activity
product_activity = df.groupby(['NumOfProducts', 'IsActiveMember'])['Exited'].agg(['count', 'mean'])
product_activity['Activity'] = product_activity.index.get_level_values(1).map({0:'Inactive', 1:'Active'})
product_activity['mean'] = product_activity['mean'].map(lambda x: f'{x:.2%}')
product_activity[['Activity', 'count', 'mean']]


## 5. Visualizations

In [ ]:
# Churn rate by geography
geo_rate = df.groupby('Geography')['Exited'].mean().sort_values(ascending=False)
geo_rate.plot(kind='bar', figsize=(8,5))
plt.title('Exit Rate by Geography')
plt.ylabel('Exit Rate')
plt.xlabel('Geography')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate by number of products
product_rate = df.groupby('NumOfProducts')['Exited'].mean()
product_rate.plot(kind='bar', figsize=(8,5))
plt.title('Exit Rate by Number of Products')
plt.ylabel('Exit Rate')
plt.xlabel('Number of Products')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate by age group
age_rate = df.groupby('AgeGroup', observed=True)['Exited'].mean()
age_rate.plot(kind='bar', figsize=(9,5))
plt.title('Exit Rate by Age Group')
plt.ylabel('Exit Rate')
plt.xlabel('Age Group')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Engagement vs churn
activity_rate = df.groupby('IsActiveMember')['Exited'].mean()
activity_rate.index = ['Inactive', 'Active']
activity_rate.plot(kind='bar', figsize=(7,5))
plt.title('Exit Rate: Active vs Inactive Customers')
plt.ylabel('Exit Rate')
plt.xlabel('Customer Status')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Customer Risk Segmentation

In [ ]:
# Business-oriented risk rules
df['RiskSegment'] = np.select(
    [
        (df['Exited'] == 1),
        (df['IsActiveMember'] == 0) & (df['NumOfProducts'] == 1),
        (df['Geography'] == 'Germany') & (df['IsActiveMember'] == 0),
        (df['Age'].between(46, 65)) & (df['IsActiveMember'] == 0)
    ],
    [
        'Already Exited',
        'High Risk - Inactive + 1 Product',
        'High Risk - Germany + Inactive',
        'High Risk - Older + Inactive'
    ],
    default='Lower Priority / Retain'
)

risk_summary = df['RiskSegment'].value_counts().to_frame('Customers')
risk_summary['Share'] = risk_summary['Customers'] / len(df)
risk_summary['Share'] = risk_summary['Share'].map(lambda x: f'{x:.2%}')
risk_summary


## 7. Key Findings from This Dataset

### Findings

- **Overall exit rate is 20.37%**: 2,037 of 10,000 customers have exited.
- **Germany is the highest-risk geography**, with an exit rate of about **32.44%**, compared with about 16%–17% in France and Spain.
- **Inactive customers are materially more likely to exit**: about **26.85%** versus **14.27%** for active members.
- **Product count is a major signal**: customers with 1 product have about **27.71%** exit rate, while customers with 2 products have only about **7.58%**. Customers with 3–4 products show extremely high exit rates, but these groups are very small and should be investigated rather than generalized.
- **Age is important**: the 46–55 and 56–65 groups show exit rates around **50.57%** and **48.32%**, respectively.
- **Germany + inactivity is especially risky**: the observed exit rate is about **41.08%** for inactive German customers.
- **Engagement is a strong retention lever**: active membership is associated with substantially lower exit rates across countries and product groups.

> Important analytical note: this is observational analysis. The relationships above are useful retention signals, but they do not prove that a specific action causes churn to fall.


## 8. Retention Strategy

| Priority | Customer segment | Recommended action |
|---|---|---|
| 1 | Inactive customers with 1 product | Personalized engagement campaign, product education, usage reminders |
| 2 | German customers, especially inactive | Dedicated retention offers and relationship-manager outreach |
| 3 | Customers aged 46–65 | Proactive service check-ins, loyalty benefits, personalized financial products |
| 4 | Customers with 3–4 products | Investigate why churn is unusually high; review product fit, fees, service issues |
| 5 | Active customers | Loyalty program and cross-sell opportunities to protect retention |

### Suggested retention KPIs
- Monthly/quarterly exit rate
- Active-member rate
- Average products per customer
- Exit rate for 1-product customers
- Exit rate by geography
- Exit rate by age group
- Reactivation rate of inactive customers
- Retention campaign conversion rate


## 9. Optional Predictive Model

In [ ]:
# A simple classification model can be added after the descriptive analysis.
# This section intentionally uses a train/test split and one-hot encoding.

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model_df = pd.read_csv('European_Bank.csv')

X = model_df.drop(columns=['Exited', 'Surname', 'CustomerId'])
y = model_df['Exited']

categorical = ['Geography', 'Gender']
numeric = [c for c in X.columns if c not in categorical]

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical)
])

model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))


## 10. Project Conclusion

This project converts customer-level banking data into a retention-focused analytical framework. The strongest observable warning signals are **inactivity, geography (particularly Germany), age, and product utilization**. The business should prioritize inactive one-product customers, investigate the German market separately, and treat the 3–4 product segment as an investigation cohort because its sample size is small.

### Final business objective
Use these signals to create a monthly **Customer Retention Dashboard** and a prioritized list of at-risk customers for targeted outreach.
